# BEH 40: Motor renewal cycles and renewal history

Behaviour read per Motor renewal cycle (`beh.MotorRenewalCycle`: one row per offered Motor renewal), then carried to
every customer month-end as the last completed renewal (`LR_*`) and the last three (`L3R_*`) in
`beh.CustomerMonthRenewalHistory`. A cycle counts as completed at a month-end once its due date plus 30 days has
passed, so nothing after the month-end is read.

Events and inbound calls attach to the customer's Motor cycle whose due date is nearest, within due minus 90 to plus
30 days (the ads_v2 call rule, D-E1.24): ads_v2 CycleAnchorDate matches PolicyRenewal for only 50-65% of renewal
events and renewal calls carry no policy code. Re-quote fields need quote coverage (GDPR wipe before 2026-05-06), so
they are null for earlier cycles. Route, reassurance and proactivity rest on call events (R2b, R3b), so a cycle's
events count as covered (EventWindowCovered) only from the call start (cov_calls, Nov 2022).

RenewalRoute comes from the channel PolicyRenewal says the renewal was processed through (SubChannel 'e) Online' is
online; Back Office and Web Assist are assisted; the D-E1.26 split) and whether the customer phoned in the cycle
(R2b or an inbound call linked by phone). The event codes cannot give the route: ads_v2 writes R3b ("renews on an
inbound call") beside R3a ("renews online") on most online renewals.

Event order uses only usable times (stg.beh_event.TimeUsable): ads_v2 places R3a / R3c / R3d (renews online, on an
outbound call, on chat) and R1.B2 at derived times (due minus 60, or the due date), so they cannot be ordered against
a prompt. R0 (pre-renewal notice, sent by rule at T-60) keeps its rule time and still counts in PromptsBeforeAction.
Proactivity is ActedBeforeInvite: the first customer action before the first invitation (R1a / R1b), as Vijay
decided on 30 Oct (the curation's "before the first prompt" would always be read against R0 at T-60).

**If a cell fails**

1. Read the description above the failing cell: what it builds, what it reads, its grain and key.
2. Fix the cause (usually a renamed or missing source column, or a broken key), then rerun from that cell.
   Every cell is `create or replace`, so rerunning is safe.
3. The checks cell at the end names any table that is empty or has duplicate / null keys, and the likely cause.

Run order: BEH_00_Setup is run by every notebook. Then 10, 20, 30, 40, 50, 60, in that order.


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run BEH_00_Setup

In [ ]:
%%pyspark
beh_run_log("BEH_40_RenewalHistory", "started")


## beh_motor_cycle


**`stg.beh_motor_cycle`**

Offered Motor renewal cycles due up to renewal_outcome_end, with the offer, prices, promo code, the retained flag and the channel the renewal was processed through. PriceMovement is null without an invite or last-year premium, or outside -75% to +300% (data errors: 0.2% of invites are above +300%). The warehouse's own price movement (RenEUR\*ChgIncFee, GWPYoY\*, PriceIncrease\*) is carried beside it as ods_v2 calculates it with the DWH definition.

- **Reads:** stg.beh_renewal_due, ods_v2.PolicyRenewal, stg.beh_motor_retention
- **Key:** PolicyCode + DueDate

In [ ]:
create or replace table ${beh.stg}.beh_motor_cycle as
select  d.SCV_Customer_Key,
        d.PolicyCode,
        d.DueDate,
        d.MonthNo,
        case when t.RetentionSignal is not null then 1 else 0 end                   as Retained,
        t.RetentionSignal,
        r.LYEURGrossPremium,
        r.RenEURPremInvite,
        r.RenEURPremOffer,
        r.TYEURGrossPremium,
        r.RenEURFee,
        r.TYEURFees,
        case when r.RenEURPremInvite > 0 and r.LYEURGrossPremium > 0
              and (r.RenEURPremInvite - r.LYEURGrossPremium) / r.LYEURGrossPremium between -0.75 and 3
             then (r.RenEURPremInvite - r.LYEURGrossPremium) / r.LYEURGrossPremium end as PriceMovement,
        r.RenEURInviteChgIncFee,
        r.RenEUROfferChgIncFee,
        r.GWPYoYCustChg,
        r.GWPYoYCustPct,
        r.GWPYoYInsurerChg,
        r.GWPYoYInsurerPct,
        r.PriceIncrease,
        r.PriceIncreasePC,
        r.PriceIncreaseOffer,
        r.PromoCodeApplied,
        r.PromoCodeTreatment,
        r.Channel                                                                   as ProcessingChannel,
        r.SubChannel                                                                as ProcessingSubChannel
from    ${beh.stg}.beh_renewal_due d
join    ${beh.ods}.policyrenewal r
        on  r.PolicyCode = d.PolicyCode
        and r.LYPolicyRenewDateAdj = d.DueDate
left join ${beh.stg}.beh_motor_retention t
        on  t.PolicyCode = d.PolicyCode
        and t.DueDate = d.DueDate
where   d.PolicyTypeGroup = 'Motor'
  and   d.PolicyOfferNum = 1
  and   d.DueDate <= to_date('${beh.renewal_outcome_end}')


## beh_motor_cycle_event


**`stg.beh_motor_cycle_event`**

Motor events attached to the customer's nearest Motor renewal cycle (due minus 90 to plus 30 days), with the time of the cycle's first customer action. CycleCandidates \> 1 marks events that fell in two cycles' windows (customers with several Motor policies).

- **Reads:** stg.beh_event, stg.beh_motor_cycle
- **Key:** EventId

In [ ]:
create or replace table ${beh.stg}.beh_motor_cycle_event as
select  x.*,
        min(case when x.ren_cust_actions = 1 and x.TimeUsable = 1 then x.EventDateTime end)
            over (partition by x.PolicyCode, x.DueDate)                     as FirstActionTime
from    (select  c.PolicyCode,
                 c.DueDate,
                 e.EventId,
                 e.EventDateTime,
                 e.EventTypeId,
                 e.prompts,
                 e.is_invite,
                 e.ren_cust_actions,
                 e.friction_events,
                 e.concessions,
                 e.failed_payments,
                 e.failed_logins,
                 e.renewal_requotes,
                 e.online_quotes_motor,
                 e.TimeUsable,
                 count(*) over (partition by e.EventId)                     as CycleCandidates,
                 row_number() over (partition by e.EventId
                                    order by abs(datediff(e.EventDate, c.DueDate)), c.DueDate, c.PolicyCode) as rn
         from    ${beh.stg}.beh_event e
         join    ${beh.stg}.beh_motor_cycle c
                 on  c.SCV_Customer_Key = e.SCV_Customer_Key
                 and e.EventDate between date_sub(c.DueDate, 90) and date_add(c.DueDate, 30)
         where   e.ProductType = 'motor') x
where   x.rn = 1


## beh_motor_cycle_call


**`stg.beh_motor_cycle_call`**

Inbound and outbound voice conversations attached to the customer's nearest Motor renewal cycle (due minus 90 to plus 30 days). A call is not tied to a product, so a call about another product in that window counts.

- **Reads:** stg.beh_call_scv, stg.beh_motor_cycle
- **Key:** ConversationId

In [ ]:
create or replace table ${beh.stg}.beh_motor_cycle_call as
select  ConversationId,
        PolicyCode,
        DueDate,
        CallStart,
        Direction
from    (select  k.ConversationId,
                 c.PolicyCode,
                 c.DueDate,
                 k.CallStart,
                 k.Direction,
                 row_number() over (partition by k.ConversationId
                                    order by abs(datediff(k.CallDate, c.DueDate)), c.DueDate, c.PolicyCode) as rn
         from    ${beh.stg}.beh_call_scv k
         join    ${beh.stg}.beh_motor_cycle c
                 on  c.SCV_Customer_Key = k.SCV_Customer_Key
                 and k.CallDate between date_sub(c.DueDate, 90) and date_add(c.DueDate, 30)) x
where   rn = 1


## beh_cycle_disruption


**`stg.beh_cycle_disruption`**

Disruption in the term before each offered Motor renewal (curation #50), up to the invitation (due minus 60 days) so it is known before the renewal: a claim dated in the term (PolicyClaimMotor, the claims declared on the policy, so a claim may only appear at the next change) and an arrears step for the customer in the term (null before cov_arrears). The price rise is PriceRise5 on the cycle.

- **Reads:** stg.beh_motor_cycle, ods_v2.PolicyClaimMotor, stg.beh_event
- **Key:** PolicyCode + DueDate

In [ ]:
create or replace table ${beh.stg}.beh_cycle_disruption as
select  c.PolicyCode,
        c.DueDate,
        case when exists_claim.PolicyCode is not null then 1 else 0 end            as DisruptionClaim,
        case when date_sub(c.DueDate, 365) < to_date('${beh.cov_arrears}') then null
             when a.PolicyCode is not null then 1 else 0 end                       as DisruptionArrears
from    ${beh.stg}.beh_motor_cycle c
left join (select  distinct c2.PolicyCode, c2.DueDate
           from    ${beh.stg}.beh_motor_cycle c2
           join    ${beh.ods}.policyclaimmotor m
                   on  m.PolicyCode = c2.PolicyCode
                   and m.ClaimDate > date_sub(c2.DueDate, 365)
                   and m.ClaimDate <= date_sub(c2.DueDate, 60)) exists_claim
        on  exists_claim.PolicyCode = c.PolicyCode
        and exists_claim.DueDate = c.DueDate
left join (select  distinct c3.PolicyCode, c3.DueDate
           from    ${beh.stg}.beh_motor_cycle c3
           join    ${beh.stg}.beh_event e
                   on  e.SCV_Customer_Key = c3.SCV_Customer_Key
                   and e.arrears_events = 1
                   and e.ActivityDate > date_sub(c3.DueDate, 365)
                   and e.ActivityDate <= date_sub(c3.DueDate, 60)) a
        on  a.PolicyCode = c.PolicyCode
        and a.DueDate = c.DueDate


## beh_cycle_portal_pay


**`stg.beh_cycle_portal_pay`**

Motor renewal cycles with a successful payment in the renewal portal on the policy, from 60 days before the due date to 30 days after it (curation #1).

- **Reads:** stg.beh_motor_cycle, ods_v2.RenewalPortalEvent
- **Key:** PolicyCode + DueDate

In [ ]:
create or replace table ${beh.stg}.beh_cycle_portal_pay as
select  c.PolicyCode,
        c.DueDate,
        min(p.EventDateTime)                        as FirstPortalPayment
from    ${beh.stg}.beh_motor_cycle c
join    ${beh.ods}.renewalportalevent p
        on  upper(trim(p.PolicyCode)) = c.PolicyCode
        and p.EventType = 'PaymentSuccess'
        and to_date(p.EventDateTime) between date_sub(c.DueDate, 60) and date_add(c.DueDate, 30)
group by c.PolicyCode, c.DueDate


## MotorRenewalCycle


**`beh.MotorRenewalCycle`**

One row per offered Motor renewal: proactivity (prompts before the first customer action, acted before the first prompt, days from invitation to action), renewal route, reassurance call, concession, re-quote, price response, friction, calls, outcome and processing channel. \*WindowCovered = 0 means the source did not cover the whole cycle window; the features resting on it are null.

- **Reads:** stg.beh_motor_cycle, stg.beh_motor_cycle_event, stg.beh_motor_cycle_call, stg.beh_cycle_disruption, stg.beh_cycle_portal_pay, stg.beh_friction_v3
- **Grain:** Motor renewal cycle (policy code x due date)
- **Key:** PolicyCode + DueDate

In [ ]:
create or replace table ${beh.target}.MotorRenewalCycle as
select  c.SCV_Customer_Key,
        c.PolicyCode,
        c.DueDate,
        c.MonthNo                                                                   as DueMonthNo,
        c.EventWindowCovered,
        c.QuoteWindowCovered,
        c.CallWindowCovered,
        c.Retained,
        c.RetentionSignal,
        c.LYEURGrossPremium,
        c.RenEURPremInvite,
        c.RenEURPremOffer,
        c.TYEURGrossPremium,
        c.PriceMovement,
        case when c.PriceMovement >= 0.05 then 1 when c.PriceMovement is not null then 0 end as PriceRise5,
        case when c.PriceMovement >= 0.05 then c.Retained end                       as RenewedAfterPriceRise,
        case when c.Retained = 1 and c.RenEURPremOffer > 0 and c.TYEURGrossPremium > 0
              and (c.TYEURGrossPremium - c.RenEURPremOffer) / c.RenEURPremOffer between -0.75 and 3
             then (c.TYEURGrossPremium - c.RenEURPremOffer) / c.RenEURPremOffer end as PaidVsOffered,
        c.RenEURInviteChgIncFee,
        c.RenEUROfferChgIncFee,
        c.GWPYoYCustChg,
        c.GWPYoYCustPct,
        c.GWPYoYInsurerChg,
        c.GWPYoYInsurerPct,
        c.PriceIncrease,
        c.PriceIncreasePC,
        c.PriceIncreaseOffer,
        c.PromoCodeApplied,
        c.PromoCodeTreatment,
        c.ProcessingChannel,
        c.ProcessingSubChannel,
        e.FirstPromptTime,
        e.FirstInviteTime,
        e.FirstActionTime,
        case when c.EventWindowCovered = 1 then coalesce(e.PromptsBeforeAction, 0) end as PromptsBeforeAction,
        case when c.EventWindowCovered = 0 then null
             when e.FirstInviteTime is null then null
             when e.FirstActionTime is null then 0
             when e.FirstActionTime < e.FirstInviteTime then 1
             else 0 end                                                             as ActedBeforeInvite,
        datediff(to_date(e.FirstActionTime), to_date(e.FirstInviteTime))            as DaysInviteToAction,
        case when c.EventWindowCovered = 0 then null
             when c.Retained = 1 and c.RetentionSignal <> '1 warehouse flag'
                  then 'Retained outside the warehouse flag'
             when c.Retained = 1 and c.ProcessingSubChannel = 'e) Online'
              and not (coalesce(e.HasInboundCall, 0) = 1 or coalesce(k.InboundCalls, 0) > 0)
                  then 'Online, no call'
             when c.Retained = 1 and c.ProcessingSubChannel = 'e) Online' then 'Online after a call'
             when c.Retained = 1 and (coalesce(e.HasInboundCall, 0) = 1 or coalesce(k.InboundCalls, 0) > 0)
                  then 'Assisted, customer phoned'
             when c.Retained = 1 then 'Assisted, no inbound call'
             when coalesce(e.HasInboundCall, 0) = 1 or coalesce(k.InboundCalls, 0) > 0
                  then 'Not renewed, customer phoned'
             when e.HasPortal = 1 then 'Not renewed, online contact only'
             else 'Not renewed, no contact' end                                     as RenewalRoute,
        case when c.EventWindowCovered = 0 or c.Retained = 0 then null
             when e.FirstInboundCallTime < coalesce(e.AcceptTime, cast(c.DueDate as timestamp)) then 1
             else 0 end                                                             as ReassuranceCall,
        case when c.Retained = 1 and c.RetentionSignal = '1 warehouse flag' then case when c.TYEURFees < c.RenEURFee then 1 else 0 end end as Concession,
        case when c.Retained = 1 and c.TYEURFees < c.RenEURFee then c.RenEURFee - c.TYEURFees end as ConcessionFeeEUR,
        case when c.EventWindowCovered = 0 or c.Retained = 0 or c.RetentionSignal <> '1 warehouse flag' then null
             when c.TYEURFees < c.RenEURFee
              and to_date(least(e.FirstInboundCallTime, k.FirstCallStart)) <= to_date(coalesce(e.AcceptTime, cast(c.DueDate as timestamp)))
                  then 1
             else 0 end                                                             as ConcessionAfterContact,
        c.RenEURFee,
        c.TYEURFees,
        case when c.QuoteWindowCovered = 1 then coalesce(e.Requote, 0) end          as Requote,
        case when c.EventWindowCovered = 1 then coalesce(e.FrictionEvents, 0) end   as FrictionEvents,
        case when c.EventWindowCovered = 1 then coalesce(e.FailedPayments, 0) end   as FailedPayments,
        case when c.EventWindowCovered = 1 then coalesce(e.FailedLogins, 0) end     as FailedLogins,
        case when c.CallWindowCovered = 1 then coalesce(k.InboundCalls, 0) end      as InboundCalls,
        case when c.CallWindowCovered = 1 then coalesce(k.OutboundCalls, 0) end     as OutboundCalls,
        case when c.EventWindowCovered = 0 or e.FirstInviteTime is null then null
             when c.Retained = 1 and c.ProcessingSubChannel = 'e) Online'
              and pp.FirstPortalPayment is not null
              and e.FirstPortalLoginTime >= e.FirstInviteTime
              and not (coalesce(e.HasInboundCall, 0) = 1 or coalesce(k.InboundCalls, 0) > 0)
              and coalesce(k.OutboundCalls, 0) = 0
             then 1 else 0 end                                                      as RenewedOnOfferNoContact,
        case when c.EventWindowCovered = 0 or e.FirstInviteTime is null or c.QuoteWindowCovered = 0 then null
             when c.Retained = 1 and c.ProcessingSubChannel = 'e) Online'
              and pp.FirstPortalPayment is not null
              and e.FirstPortalLoginTime >= e.FirstInviteTime
              and not (coalesce(e.HasInboundCall, 0) = 1 or coalesce(k.InboundCalls, 0) > 0)
              and coalesce(k.OutboundCalls, 0) = 0
              and coalesce(e.Requote, 0) = 0 and coalesce(e.OnlineQuote, 0) = 0
             then 1 else 0 end                                                      as RenewedOnOfferNoContactNoQuote,
        case when c.FrictionV3Covered = 1 then v.Eligible end                       as FrictionV3Eligible,
        case when c.FrictionV3Covered = 1 then v.Affected end                       as FrictionV3Affected,
        case when c.FrictionV3Covered = 1 then v.Affected / nullif(v.Eligible, 0) end as FrictionV3Rate,
        case when c.FrictionV3Covered = 1 then v.Resolved / nullif(v.ResKnown, 0) end as FrictionV3ResolvedShare,
        coalesce(e.CycleEvents, 0)                                                  as CycleEvents,
        coalesce(e.AmbiguousCycleEvents, 0)                                         as AmbiguousCycleEvents,
        d.DisruptionClaim,
        d.DisruptionArrears,
        case when c.PriceMovement >= 0.05 then 1 when c.PriceMovement is not null then 0 end as DisruptionPriceRise,
        case when d.DisruptionClaim = 1 or d.DisruptionArrears = 1 or c.PriceMovement >= 0.05 then 1
             else 0 end                                                             as Disrupted
from    (select  m.*,
                 case when date_sub(m.DueDate, 90) >= greatest(to_date('${beh.cov_events}'), to_date('${beh.cov_calls}'))
                      then 1 else 0 end                                                       as EventWindowCovered,
                 case when date_sub(m.DueDate, 90) >= to_date('${beh.cov_quotes}') then 1 else 0 end as QuoteWindowCovered,
                 case when date_sub(m.DueDate, 90) >= to_date('${beh.cov_calls}') then 1 else 0 end  as CallWindowCovered,
                 case when date_sub(m.DueDate, 90) >= to_date('${beh.cov_friction_v3}') then 1 else 0 end as FrictionV3Covered
         from    ${beh.stg}.beh_motor_cycle m) c
left join (select  PolicyCode,
                   DueDate,
                   min(case when prompts = 1 and TimeUsable = 1 then EventDateTime end)        as FirstPromptTime,
                   min(case when is_invite = 1 and TimeUsable = 1 then EventDateTime end)      as FirstInviteTime,
                   min(FirstActionTime)                                                         as FirstActionTime,
                   sum(case when prompts = 1 and TimeUsable = 1 and (FirstActionTime is null or EventDateTime < FirstActionTime)
                            then 1 else 0 end)                                                  as PromptsBeforeAction,
                   max(case when EventTypeId in ('R2', 'R2a') then 1 else 0 end)                   as HasPortal,
                   min(case when EventTypeId in ('R2', 'R2a') and TimeUsable = 1 then EventDateTime end) as FirstPortalLoginTime,
                   max(online_quotes_motor)                                                     as OnlineQuote,
                   max(case when EventTypeId in ('R2b') then 1 else 0 end)                     as HasInboundCall,
                   min(case when EventTypeId in ('R2b') then EventDateTime end)                as FirstInboundCallTime,
                   max(case when EventTypeId in ('R3a') then 1 else 0 end)                      as RenewedOnline,
                   max(case when EventTypeId in ('R3b') then 1 else 0 end)                      as RenewedInbound,
                   max(case when EventTypeId in ('R3c') then 1 else 0 end)                     as RenewedOutbound,
                   max(case when EventTypeId in ('R3d') then 1 else 0 end)                     as RenewedChat,
                   min(case when EventTypeId in ('R3a', 'R3b', 'R3c', 'R3d') and TimeUsable = 1 then EventDateTime end) as AcceptTime,
                   max(renewal_requotes)                                                        as Requote,
                   sum(friction_events)                                                         as FrictionEvents,
                   sum(failed_payments)                                                         as FailedPayments,
                   sum(failed_logins)                                                           as FailedLogins,
                   count(*)                                                                     as CycleEvents,
                   sum(case when CycleCandidates > 1 then 1 else 0 end)                         as AmbiguousCycleEvents
           from    ${beh.stg}.beh_motor_cycle_event
           group by PolicyCode, DueDate) e
        on  e.PolicyCode = c.PolicyCode
        and e.DueDate = c.DueDate
left join (select PolicyCode, DueDate,
                  sum(case when Direction = 'inbound' then 1 else 0 end)        as InboundCalls,
                  sum(case when Direction = 'outbound' then 1 else 0 end)       as OutboundCalls,
                  min(case when Direction = 'inbound' then CallStart end)       as FirstCallStart
           from   ${beh.stg}.beh_motor_cycle_call
           group by PolicyCode, DueDate) k
        on  k.PolicyCode = c.PolicyCode
        and k.DueDate = c.DueDate
left join ${beh.stg}.beh_cycle_disruption d
        on  d.PolicyCode = c.PolicyCode
        and d.DueDate = c.DueDate
left join ${beh.stg}.beh_cycle_portal_pay pp
        on  pp.PolicyCode = c.PolicyCode
        and pp.DueDate = c.DueDate
left join (select PolicyCode, DueDate, sum(Eligible) as Eligible, sum(Affected) as Affected,
                  sum(ResKnown) as ResKnown, sum(Resolved) as Resolved
           from   ${beh.stg}.beh_friction_v3
           where  EpisodeGroup = 'renewal' and Product = 'Car'
           group by PolicyCode, DueDate) v
        on  v.PolicyCode = c.PolicyCode
        and v.DueDate = c.DueDate


## beh_cycle_asat


**`stg.beh_cycle_asat`**

For each customer month-end, the customer's Motor renewal cycles completed by then (due plus 30 days on or before the month-end) within the last 48 months, newest first (rn = 1 is the last renewal).

- **Reads:** beh.CustomerMonth, beh.MotorRenewalCycle
- **Key:** SCV_Customer_Key + MonthEndDate + PolicyCode + DueDate

In [ ]:
create or replace table ${beh.stg}.beh_cycle_asat as
select  m.SCV_Customer_Key,
        m.MonthEndDate,
        r.PolicyCode,
        r.DueDate,
        r.Retained, r.RenewalRoute, r.PromptsBeforeAction, r.ActedBeforeInvite, r.RenewedOnOfferNoContact, r.DaysInviteToAction, r.ReassuranceCall, r.Concession, r.ConcessionAfterContact, r.Requote, r.PriceMovement, r.PriceRise5, r.RenewedAfterPriceRise, r.PaidVsOffered, r.GWPYoYCustPct, r.PriceIncreasePC, r.FrictionEvents, r.FailedPayments, r.FailedLogins, r.InboundCalls, r.ProcessingSubChannel, r.FrictionV3Rate, r.FrictionV3ResolvedShare,
        row_number() over (partition by m.SCV_Customer_Key, m.MonthEndDate
                           order by r.DueDate desc, r.PolicyCode)          as rn
from    ${beh.target}.CustomerMonth m
join    ${beh.target}.MotorRenewalCycle r
        on  r.SCV_Customer_Key = m.SCV_Customer_Key
        and date_add(r.DueDate, 30) <= m.MonthEndDate
        and r.DueDate > add_months(m.MonthEndDate, -48)


## CustomerMonthRenewalHistory


**`beh.CustomerMonthRenewalHistory`**

Motor renewal behaviour as known at each customer month-end: the last completed renewal (LR\_\*) and the last three (L3R\_\*; shares are null below two observed cycles). Null where the customer has no completed Motor renewal in the last 48 months.

- **Reads:** beh.CustomerMonth, stg.beh_cycle_asat
- **Grain:** customer x month-end (as beh.CustomerMonth)
- **Key:** SCV_Customer_Key + MonthEndDate

In [ ]:
create or replace table ${beh.target}.CustomerMonthRenewalHistory as
    select  m.SCV_Customer_Key,
            m.MonthEndDate,
            count(h.PolicyCode)                                                         as MotorRenewals48m,
            max(case when h.rn = 1 then h.DueDate end)                                  as LR_DueDate,
            max(case when h.rn = 1 then h.Retained end)          as LR_Retained,
max(case when h.rn = 1 then h.RenewalRoute end)      as LR_RenewalRoute,
max(case when h.rn = 1 then h.PromptsBeforeAction end) as LR_PromptsBeforeAction,
max(case when h.rn = 1 then h.ActedBeforeInvite end) as LR_ActedBeforeInvite,
max(case when h.rn = 1 then h.RenewedOnOfferNoContact end) as LR_RenewedOnOfferNoContact,
max(case when h.rn = 1 then h.DaysInviteToAction end) as LR_DaysInviteToAction,
max(case when h.rn = 1 then h.ReassuranceCall end)   as LR_ReassuranceCall,
max(case when h.rn = 1 then h.Concession end)        as LR_Concession,
max(case when h.rn = 1 then h.ConcessionAfterContact end) as LR_ConcessionAfterContact,
max(case when h.rn = 1 then h.Requote end)           as LR_Requote,
max(case when h.rn = 1 then h.PriceMovement end)     as LR_PriceMovement,
max(case when h.rn = 1 then h.PriceRise5 end)        as LR_PriceRise5,
max(case when h.rn = 1 then h.RenewedAfterPriceRise end) as LR_RenewedAfterPriceRise,
max(case when h.rn = 1 then h.PaidVsOffered end)     as LR_PaidVsOffered,
max(case when h.rn = 1 then h.GWPYoYCustPct end)     as LR_GWPYoYCustPct,
max(case when h.rn = 1 then h.PriceIncreasePC end)   as LR_PriceIncreasePC,
max(case when h.rn = 1 then h.FrictionEvents end)    as LR_FrictionEvents,
max(case when h.rn = 1 then h.FailedPayments end)    as LR_FailedPayments,
max(case when h.rn = 1 then h.FailedLogins end)      as LR_FailedLogins,
max(case when h.rn = 1 then h.InboundCalls end)      as LR_InboundCalls,
max(case when h.rn = 1 then h.ProcessingSubChannel end) as LR_ProcessingSubChannel,
max(case when h.rn = 1 then h.FrictionV3Rate end)    as LR_FrictionV3Rate,
max(case when h.rn = 1 then h.FrictionV3ResolvedShare end) as LR_FrictionV3ResolvedShare,
            sum(case when h.rn <= 3 then 1 else 0 end)                                  as L3R_Renewals,
            case when sum(case when h.rn <= 3 and h.ActedBeforeInvite is not null then 1 else 0 end) >= 2
                 then avg(case when h.rn <= 3 then h.ActedBeforeInvite end) end         as L3R_ProactivityShare,
            case when sum(case when h.rn <= 3 and h.Concession is not null then 1 else 0 end) >= 1
                 then sum(case when h.rn <= 3 then h.Concession end) end                as L3R_Concessions,
            case when sum(case when h.rn <= 3 and h.ConcessionAfterContact is not null then 1 else 0 end) >= 1
                 then sum(case when h.rn <= 3 then h.ConcessionAfterContact end) end    as L3R_ConcessionsAfterContact,
            case when sum(case when h.rn <= 3 and h.Requote is not null then 1 else 0 end) >= 2
                 then avg(case when h.rn <= 3 then h.Requote end) end                   as L3R_RequoteShare,
            case when sum(case when h.rn <= 3 then 1 else 0 end) >= 2
                 then avg(case when h.rn <= 3 then h.Retained end) end                  as L3R_RetainedShare,
            avg(case when h.rn <= 3 then h.PromptsBeforeAction end)                     as L3R_AvgPromptsBeforeAction,
            case when sum(case when h.rn <= 3 and h.RenewalRoute is not null then 1 else 0 end) >= 2
                 then count(distinct case when h.rn <= 3 then h.RenewalRoute end) end   as L3R_DistinctRoutes
    from    ${beh.target}.CustomerMonth m
    left join ${beh.stg}.beh_cycle_asat h
            on  h.SCV_Customer_Key = m.SCV_Customer_Key
            and h.MonthEndDate = m.MonthEndDate
    group by m.SCV_Customer_Key, m.MonthEndDate


## beh_mfq_purchase_base


**`stg.beh_mfq_purchase_base`**

MFQ quotes that became a Motor policy (ods_v2.Quote ConvertedToPolicy, linked to the policy bought from January 2024), with the customer of that policy and the purchase time (the first successful online payment, else the quote's last activity). Curation #14 to #17.

- **Reads:** ods_v2.Quote, ods_v2.QuoteMotor, stg.beh_policy_scv
- **Key:** GuidKey

In [ ]:
create or replace table ${beh.stg}.beh_mfq_purchase_base as
select  q.SourceSystemQuoteReference                                                as GuidKey,
        min(l.SCV_Customer_Key)                                                     as SCV_Customer_Key,
        min(trim(q.SourcePolicyCode))                                               as PolicyCode,
        min(coalesce(m.FirstSuccessfulPaymentDateTimeUtc, m.LastCreatedDateTimeUtc)) as PurchaseTs
from    ${beh.ods}.quote q
join    ${beh.stg}.beh_policy_scv l on l.PolicyCode = trim(q.SourcePolicyCode)
join    ${beh.ods}.quotemotor m on m.GuidKey = q.SourceSystemQuoteReference
where   q.SourceSystemId = 2
  and   q.ConvertedToPolicy = 1
  and   nullif(trim(q.SourcePolicyCode), '') is not null
  and   to_date(coalesce(m.FirstSuccessfulPaymentDateTimeUtc, m.LastCreatedDateTimeUtc))
        between to_date('${beh.cov_mfq_purchase}') and to_date('${beh.last_me}')
group by q.SourceSystemQuoteReference


## beh_mfq_purchase_anc: the retrieval chain behind each purchase


In [ ]:
%%pyspark
# The full MFQ retrieval chain behind each purchased quote (curation #14 to #17): walk RetrievedGuidKey back from the
# purchased quote one step at a time until no new quote is found (chains reach 10 or more steps; capped at 40).
from pyspark.sql import functions as F
ret = spark.table(f"{stg_schema}.beh_mfq_retrieval").select("GuidKey", "RetrievedGuidKey").distinct()
frontier = spark.table(f"{stg_schema}.beh_mfq_purchase_base").select(
    F.col("GuidKey").alias("PurchaseGuidKey"), F.col("GuidKey").alias("Cur"))
seen = frontier.select("PurchaseGuidKey", F.col("Cur").alias("AncGuidKey"))
parts = []
for level in range(1, 41):
    nxt = (frontier.join(ret, frontier.Cur == ret.GuidKey)
           .select("PurchaseGuidKey", F.col("RetrievedGuidKey").alias("AncGuidKey"))
           .join(seen, ["PurchaseGuidKey", "AncGuidKey"], "left_anti").distinct()
           .localCheckpoint())
    n = nxt.count()
    if n == 0:
        break
    parts.append(nxt.withColumn("Level", F.lit(level)))
    seen = seen.unionByName(nxt).localCheckpoint()
    frontier = nxt.select("PurchaseGuidKey", F.col("AncGuidKey").alias("Cur"))
    print("level", level, n)
anc = parts[0]
for x in parts[1:]:
    anc = anc.unionByName(x)
anc.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{stg_schema}.beh_mfq_purchase_anc")
print("ancestors", spark.table(f"{stg_schema}.beh_mfq_purchase_anc").count())


## beh_mfq_purchase_attr


**`stg.beh_mfq_purchase_attr`**

The answers on each purchased quote and on every quote in its retrieval chain: when it was made, the cover chosen at the selection steps (01 Comprehensive, 02 TPFT, the last one chosen), the cover and NCD protection options shown (ViewedCoverType, ViewedBonusProtectionType), voluntary excess, NCD years, KeyCare selected on any insurer response, and the entry channel (FirstMktChannel).

- **Reads:** stg.beh_mfq_purchase_base, stg.beh_mfq_purchase_anc, ods_v2.QuoteMotor, ods_v2.QuoteMotorStep, ods_v2.QuoteMotorSummary, ods_v2.QuoteMotorKeyCare
- **Key:** GuidKey

In [ ]:
create or replace table ${beh.stg}.beh_mfq_purchase_attr as
select  g.GuidKey,
        m.FirstCreatedDateTimeUtc                       as QuoteTs,
        c.ChosenCover,
        s.ViewedCoverType,
        s.ViewedBonusProtectionType,
        m.CoverVoluntaryExcess                          as Excess,
        m.NCDClaimedYearsEarned                         as NcdYears,
        coalesce(k.IsKeyCareSelected, 0)                as KeyCareSelected,
        s.FirstMktChannel
from    (select GuidKey from ${beh.stg}.beh_mfq_purchase_base
         union
         select AncGuidKey from ${beh.stg}.beh_mfq_purchase_anc) g
join    ${beh.ods}.quotemotor m on m.GuidKey = g.GuidKey
left join (select GuidKey, max_by(CoverCode, StepDateTimeUtc) as ChosenCover
           from   ${beh.ods}.quotemotorstep
           where  CoverCode in ('01', '02')
           group by GuidKey) c on c.GuidKey = g.GuidKey
left join (select GuidKey, max(ViewedCoverType) as ViewedCoverType,
                  max(ViewedBonusProtectionType) as ViewedBonusProtectionType, max(FirstMktChannel) as FirstMktChannel
           from   ${beh.ods}.quotemotorsummary
           group by GuidKey) s on s.GuidKey = g.GuidKey
left join ${beh.ods}.quotemotorkeycare k on k.GuidKey = g.GuidKey


## beh_mfq_purchase


**`stg.beh_mfq_purchase`**

One row per MFQ purchase with the agreed purchase features, comparing the purchased quote with the quotes in its retrieval chain made in the 90 days before the purchase. StandardConfig (#14): 0 when a predecessor differs in cover (the chosen cover where both quotes have one, else the cover shown), NCD protection shown, voluntary excess or NCD years; 1 otherwise, including buyers with no predecessor. ProtectionChange (#16): added, removed, both or no change, on KeyCare selected and the NCD protection shown; KeyCareBought from the sale. EntryChannel (#15): generic (PPC or SEO Generic), branded (PPC or SEO Brand), direct and other, or unknown. ReturnGapDays (#17): days from the earliest predecessor to the purchase (0 without one); Returned7d = 1 at 7 days or more.

- **Reads:** stg.beh_mfq_purchase_base, stg.beh_mfq_purchase_anc, stg.beh_mfq_purchase_attr, ods_v2.PolicySale
- **Key:** GuidKey

In [ ]:
create or replace table ${beh.stg}.beh_mfq_purchase as
select  b.GuidKey,
        b.SCV_Customer_Key,
        b.PolicyCode,
        b.PurchaseTs,
        to_date(b.PurchaseTs)                                                       as PurchaseDate,
        count(x.GuidKey)                                                            as PredecessorQuotes,
        case when max(case when (case when a.ChosenCover is not null and x.ChosenCover is not null
                                      then a.ChosenCover <> x.ChosenCover
                                      else a.ViewedCoverType <> x.ViewedCoverType end)
                             or a.ViewedBonusProtectionType <> x.ViewedBonusProtectionType
                             or a.Excess <> x.Excess
                             or a.NcdYears <> x.NcdYears then 1 else 0 end) = 1
             then 0 else 1 end                                                      as StandardConfig,
        case when max(case when (a.KeyCareSelected = 1 and x.KeyCareSelected = 0)
                             or (a.ViewedBonusProtectionType = 'Step Back and FBP'
                                 and x.ViewedBonusProtectionType <> 'Step Back and FBP') then 1 else 0 end) = 1
              and max(case when (a.KeyCareSelected = 0 and x.KeyCareSelected = 1)
                             or (a.ViewedBonusProtectionType <> 'Step Back and FBP'
                                 and x.ViewedBonusProtectionType = 'Step Back and FBP') then 1 else 0 end) = 1
                  then 'both'
             when max(case when (a.KeyCareSelected = 1 and x.KeyCareSelected = 0)
                             or (a.ViewedBonusProtectionType = 'Step Back and FBP'
                                 and x.ViewedBonusProtectionType <> 'Step Back and FBP') then 1 else 0 end) = 1
                  then 'added'
             when max(case when (a.KeyCareSelected = 0 and x.KeyCareSelected = 1)
                             or (a.ViewedBonusProtectionType <> 'Step Back and FBP'
                                 and x.ViewedBonusProtectionType = 'Step Back and FBP') then 1 else 0 end) = 1
                  then 'removed'
             else 'no change' end                                                   as ProtectionChange,
        max(a.KeyCareSelected)                                                      as KeyCareSelected,
        max(ks.KeyCareBought)                                                       as KeyCareBought,
        case when max(a.FirstMktChannel) in ('PPC Generic', 'SEO Generic') then 'generic'
             when max(a.FirstMktChannel) in ('PPC Brand', 'SEO Brand') then 'branded'
             when max(a.FirstMktChannel) is null or max(a.FirstMktChannel) = 'Not Classified' then 'unknown'
             else 'direct and other' end                                            as EntryChannel,
        max(a.FirstMktChannel)                                                      as FirstMktChannel,
        coalesce(datediff(to_date(b.PurchaseTs), to_date(min(x.QuoteTs))), 0)       as ReturnGapDays,
        case when datediff(to_date(b.PurchaseTs), to_date(min(x.QuoteTs))) >= 7 then 1 else 0 end as Returned7d
from    ${beh.stg}.beh_mfq_purchase_base b
join    ${beh.stg}.beh_mfq_purchase_attr a on a.GuidKey = b.GuidKey
left join (select n.PurchaseGuidKey, t.*
           from   ${beh.stg}.beh_mfq_purchase_anc n
           join   ${beh.stg}.beh_mfq_purchase_attr t on t.GuidKey = n.AncGuidKey) x
        on  x.PurchaseGuidKey = b.GuidKey
        and x.QuoteTs < b.PurchaseTs
        and x.QuoteTs >= b.PurchaseTs - interval 90 days
left join (select trim(PolicyCode) as PolicyCode, max(case when KeyCareFlag = 1 then 1 else 0 end) as KeyCareBought
           from   ${beh.ods}.policysale
           where  ReportingSaleType = 'New Business'
           group by trim(PolicyCode)) ks on ks.PolicyCode = b.PolicyCode
group by b.GuidKey, b.SCV_Customer_Key, b.PolicyCode, b.PurchaseTs


## CustomerMonthPurchase


**`beh.CustomerMonthPurchase`**

How the customer bought, as known at each month-end: the features of their latest MFQ purchase (LP\_\*) made by the month-end, and the number of MFQ purchases since January 2024. Null for customers who have not bought through MFQ since January 2024. Curation #14 to #17.

- **Reads:** beh.CustomerMonth, stg.beh_mfq_purchase
- **Grain:** customer x month-end (as beh.CustomerMonth)
- **Key:** SCV_Customer_Key + MonthEndDate

In [ ]:
create or replace table ${beh.target}.CustomerMonthPurchase as
select  m.SCV_Customer_Key,
        m.MonthEndDate,
        count(p.GuidKey)                                                as MfqPurchases,
        max(case when p.rn = 1 then p.PurchaseDate end)                 as LP_PurchaseDate,
        max(case when p.rn = 1 then p.PredecessorQuotes end)            as LP_PredecessorQuotes,
        max(case when p.rn = 1 then p.StandardConfig end)               as LP_StandardConfig,
        max(case when p.rn = 1 then p.ProtectionChange end)             as LP_ProtectionChange,
        max(case when p.rn = 1 then p.KeyCareSelected end)              as LP_KeyCareSelected,
        max(case when p.rn = 1 then p.KeyCareBought end)                as LP_KeyCareBought,
        max(case when p.rn = 1 then p.EntryChannel end)                 as LP_EntryChannel,
        max(case when p.rn = 1 then p.FirstMktChannel end)              as LP_FirstMktChannel,
        max(case when p.rn = 1 then p.ReturnGapDays end)                as LP_ReturnGapDays,
        max(case when p.rn = 1 then p.Returned7d end)                   as LP_Returned7d
from    ${beh.target}.CustomerMonth m
left join (select  c.SCV_Customer_Key,
                   c.MonthEndDate,
                   u.*,
                   row_number() over (partition by c.SCV_Customer_Key, c.MonthEndDate
                                      order by u.PurchaseTs desc, u.GuidKey) as rn
           from    ${beh.target}.CustomerMonth c
           join    (select GuidKey, SCV_Customer_Key as BuyerKey, PurchaseTs, PurchaseDate, PredecessorQuotes,
                           StandardConfig, ProtectionChange, KeyCareSelected, KeyCareBought, EntryChannel,
                           FirstMktChannel, ReturnGapDays, Returned7d
                    from   ${beh.stg}.beh_mfq_purchase) u
                   on  u.BuyerKey = c.SCV_Customer_Key
                   and u.PurchaseDate <= c.MonthEndDate) p
        on  p.SCV_Customer_Key = m.SCV_Customer_Key
        and p.MonthEndDate = m.MonthEndDate
group by m.SCV_Customer_Key, m.MonthEndDate


## Checks


In [ ]:
%%pyspark
# Row count, duplicate keys and null keys for every table this notebook built. Logged to BehBuildLog;
# the cell fails if any check failed, naming the table and the likely cause.
beh_check("BEH_40_RenewalHistory", [{"table": "{stg}.beh_motor_cycle", "key": ["PolicyCode", "DueDate"]}, {"table": "{stg}.beh_motor_cycle_event", "key": ["EventId"]}, {"table": "{stg}.beh_motor_cycle_call", "key": ["ConversationId"]}, {"table": "{stg}.beh_cycle_disruption", "key": ["PolicyCode", "DueDate"]}, {"table": "{stg}.beh_cycle_portal_pay", "key": ["PolicyCode", "DueDate"]}, {"table": "{target}.MotorRenewalCycle", "key": ["PolicyCode", "DueDate"]}, {"table": "{stg}.beh_cycle_asat", "key": ["SCV_Customer_Key", "MonthEndDate", "PolicyCode", "DueDate"]}, {"table": "{target}.CustomerMonthRenewalHistory", "key": ["SCV_Customer_Key", "MonthEndDate"]}, {"table": "{stg}.beh_mfq_purchase_base", "key": ["GuidKey"]}, {"table": "{stg}.beh_mfq_purchase_attr", "key": ["GuidKey"]}, {"table": "{stg}.beh_mfq_purchase", "key": ["GuidKey"]}, {"table": "{target}.CustomerMonthPurchase", "key": ["SCV_Customer_Key", "MonthEndDate"]}, {"table": "{stg}.beh_mfq_purchase_anc", "key": ["PurchaseGuidKey", "AncGuidKey"]}])


In [ ]:
%%pyspark
beh_run_log("BEH_40_RenewalHistory", "succeeded")
